# Cuaderno 01 contra la industria: texto y lenguaje

Las ideas del cuaderno `01_texto_y_lenguaje` compiten con lo que se usa hoy:

| Tu idea | Compite contra | Qué se mide |
|---|---|---|
| Tokenizador por **sílabas** (y la heurística original) | BPE (GPT, Llama), SentencePiece Unigram (T5, Gemma), el tokenizador de **GPT-4o**, palabras y caracteres | tokens por palabra, palabras desconocidas, velocidad y exactitud al clasificar reseñas |
| **Posición jerárquica** (pares de sílabas; palabra + sílaba) | posición aprendida y senoidal del Transformer | exactitud y velocidad |
| **Segmentador de sílabas** BiLSTM | reglas, pyphen (LibreOffice) y un Transformer de caracteres | palabras exactas sobre 128 palabras silabeadas según la RAE |
| **Corrector ortográfico GNN** (grafo de letras + auto-atención) | BiLSTM y Transformer de caracteres, y un corrector por diccionario (Norvig/SymSpell) | % de letras mal tras corregir, palabras correctas, velocidad |
| **Agentes de atención** (binaria con RL, por niveles con RL, hebbiana) | TextCNN, Transformer y fastText | exactitud, latencia, tokens/s y qué parte del texto leen |

Además, cada arquitectura de texto se mide por dentro: neuronas muertas, gradientes y parecido interno (CKA).

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*. Todo se guarda en `/content/`; al final descarga el zip.

## 1. Preparar (token de GitHub en *Secretos de Colab* como `GITHUB_TOKEN`; nunca en una celda)

In [ ]:
import os, subprocess, torch
from google.colab import userdata

os.environ["HF_HOME"] = "/content/hf"
os.environ["LABORATORIO_CACHE"] = "/content/cache"
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

if not os.path.exists("/content/Notbook"):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/SiriusMagic/Notbook.git", "/content/Notbook"], check=True)
%cd /content/Notbook
!pip install -q -r laboratorio/requirements.txt
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO hay GPU (irá lento)")

## 2. Tokenizadores: tus sílabas contra BPE, Unigram y GPT-4o (≈ 20–40 min con GPU)

In [ ]:
!python -m laboratorio.banco --tokenizadores --tarea resenas_es --semillas 3 --salida /content/resultados/01

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/content/resultados/01/tokenizadores.md", encoding="utf-8").read()))

## 3. Segmentadores de sílabas (≈ 5–10 min con GPU)

In [ ]:
!python -m laboratorio.banco --segmentadores --semillas 3 --salida /content/resultados/01
display(Markdown(open("/content/resultados/01/segmentadores.md", encoding="utf-8").read()))

## 3b. Corrección ortográfica: tu GNN contra BiLSTM, Transformer y diccionario (≈ 20–40 min con GPU)

Tu red de la celda 26 en tres versiones: tal cual entrenada con refuerzo (como el cuaderno), tal cual entrenada de forma supervisada, y con conexiones residuales (el arreglo).

In [ ]:
!python -m laboratorio.banco --correccion --semillas 3 --salida /content/resultados/01
display(Markdown(open("/content/resultados/01/correccion.md", encoding="utf-8").read()))

## 4. Agentes de atención y piezas contra TextCNN, Transformer y fastText (≈ 1–2 h con GPU)

Incluye tus tres agentes de atención y las piezas del cuaderno 02 dentro de cada arquitectura de texto. Para ir más rápido: `--pieza ida --pieza seno` o `--rapido`.

In [ ]:
!python -m laboratorio.banco --competencia --tipo texto --guardar /content/modelos --salida /content/resultados/01
display(Markdown(open("/content/resultados/01/competencia.md", encoding="utf-8").read()))

## 5. Descargar los resultados

In [ ]:
!cd /content && zip -qr resultados_01.zip resultados/01
from google.colab import files
files.download("/content/resultados_01.zip")